# Milestone 7 - RAG master prompt & grounded generation

The top-5 reranked child chunks are expanded to their parent page / section / row, numbered `[S1]..[Sn]` with document name and page, and inserted into a grounded system prompt. **Phi-4-mini-instruct** (llama.cpp, temperature 0.0, fixed seed) must answer only from those sources and cite every sentence.

In [1]:
import os, sys, json, time, warnings, pathlib
warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 120)
from rag import config

In [2]:
from rag import generator, store
from rag.pipeline import RAGPipeline
store.ensure_qdrant(); generator.ensure_llm()
pipe = RAGPipeline(audit=False)
print("reranker:", pipe.reranker_name)
print(generator.SYSTEM_PROMPT.format(company="Halvorsen Industrial Systems"))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

reranker: ./models/bge-reranker-large-finetuned
You are the Halvorsen Industrial Systems document assistant. You answer questions strictly from the numbered SOURCES supplied with each question.

Rules:
1. Use ONLY facts explicitly stated in the SOURCES. Never use outside knowledge, never guess, never infer values that are not written.
2. End every sentence with the citation of the source(s) that support it, written as [S1] or [S1][S2].
3. Copy numbers, units, dates, codes and names exactly as they appear in the sources.
4. Always answer with 1-3 complete sentences that restate the subject of the question. Never answer with only a number, date or name. Do not add background, advice or caveats.
5. If the SOURCES do not contain the answer, reply with exactly: INSUFFICIENT_EVIDENCE

Examples of the required style (fictional values):
QUESTION: What is the dry weight of the HX-999?
ANSWER: The dry weight of the HX-999 is 812 kg [S1].
QUESTION: Who is the account manager for Example Water Oy?

### The exact prompt for one question

In [3]:
q = "How heavy is the HX-250 when empty?"
cands = pipe.retriever.retrieve(q)
top = pipe.reranker.rerank(q, cands)
from rag.pipeline import expand_to_parents
sources = expand_to_parents(top)
print(generator.build_messages(q, sources)[1]["content"][:2500])

SOURCES:

[S1] (document: HX_Pump_Series_Technical_Specification.docx, p. 6)
## 6. HX-250 Centrifugal Pump

The HX-250 is recommended for boiler feed water applications. It uses a closed impeller and a mechanical shaft seal as standard.

Maximum flow rate: 770 L/min.
Maximum head: 133 m.
Motor power: 111.9 kW.
Impeller diameter: 152 mm.
Dry weight: 159 kg.
Inlet connection: DN80.
Maximum fluid temperature: 160 degC.
Motor enclosure rating: IP65.

QUESTION: How heavy is the HX-250 when empty?


In [4]:
out = generator.chat(generator.build_messages(q, sources))
print("raw model output:", out["text"]); print("tokens:", out["usage"], "| %.2fs" % out["latency_s"])

raw model output: The dry weight of the HX-250 is 159 kg [S1].
tokens: {'completion_tokens': 17, 'prompt_tokens': 422, 'total_tokens': 439, 'prompt_tokens_details': {'cached_tokens': 0}} | 0.49s


### End-to-end answers with rendered citations

In [5]:
questions = ["What is the maximum head of the HX-320 pump?", "How long must CCTV footage be kept?",
             "Who leads the Bilbao Plant site?", "When was CU-4 firmware 4.6 released?",
             "What caused the VPN gateway outage?", "How many days lead time does supplier Polaris Polymers have?"]
rows = []
for q in questions:
    res = pipe.answer(q)
    rows.append({"question": q, "status": res["status"], "answer": res["answer"], "seconds": round(res["timings_s"]["total"], 2)})
pd.DataFrame(rows)

,question,status,answer,seconds
0,What is the maximum head of the HX-320 pump?,verified,"The maximum head of the HX-320 is 41 m [HX_Pump_Series_Technical_Specification.docx, p. 8].",0.90
1,How long must CCTV footage be kept?,verified,"CCTV footage is retained for 30 days [HIS_Employee_Handbook_2026.pdf, p. 6].",0.78
2,Who leads the Bilbao Plant site?,verified,"The site director of Bilbao Plant is Marta Zielinski [HIS_Annual_Operations_Report_FY2025.pdf, p. 7].",1.39
3,When was CU-4 firmware 4.6 released?,verified,"Firmware 4.6 was released on 12 August 2024 [CU4_Controller_Firmware_Release_Notes.docx, p. 8].",0.83
4,What caused the VPN gateway outage?,verified,"Incident INC-2025-960 was caused by a faulty network switch firmware update [incident_postmortems.md, section 4].",0.80
5,How many days lead time does supplier Polaris Polymers have?,verified,"The lead time for Polaris Polymers is 67 days [supplier_register.csv, row 3].",1.15


### Out-of-scope questions are refused, not invented

In [6]:
for q in ["What is the CEO's annual salary?", "What is the maximum flow rate of the HX-990 pump?", "What was group revenue in FY2019?"]:
    res = pipe.answer(q)
    print(f"{q}\n  -> [{res['status']}] {res['answer']}\n")

What is the CEO's annual salary?
  -> [abstained] I cannot verify an answer to this question from the available documents.



What is the maximum flow rate of the HX-990 pump?
  -> [abstained] I cannot verify an answer to this question from the available documents.



What was group revenue in FY2019?
  -> [abstained] I cannot verify an answer to this question from the available documents.

